# Positional Encoding
Positional encoding is a technique used to inject information about the relative or absolute position of tokens in a sequence into the input embeddings. Since transformer models, such as those used in large language models (LLMs), do not have an inherent sense of order due to their attention mechanism, positional encoding provides the necessary context about the sequence's structure.

In LLMs, positional encodings are added to the token embeddings before being passed into the transformer layers. These encodings can be learned (trainable parameters) or fixed (e.g., sinusoidal functions). They enable the model to understand the order of words or tokens, which is crucial for tasks like language modeling, translation, and text generation.

Without positional encoding, the model would treat the input as a bag of words, losing the sequential nature of language. This would significantly degrade the model's ability to capture syntactic and semantic relationships, making it less effective in understanding and generating coherent text.

![positional-encoding](./resources/positional-encoding.png)

In [29]:
import math

def positional_encoding(pos: int, index: int, dimension: int):
    """
    Return the positional encoding given:
    - The position of the token
    - The index of the vector
    - The dimension of the embedding
    """
    if index % 2 == 0:
        return math.cos(pos / (10000 ** (2 * index / dimension)))
    else:
        return math.sin(pos / (10000 ** (2 * index / dimension)))

example_embedding = [
    [0, 1.5, 1, 0],
    [2, 0.5, 0.8, 1],
    [3, 1, 0.85, 0.11],
]

for pos in range(len(example_embedding)):
    pe = []
    for index in range(len(example_embedding[pos])):
        pe.append(positional_encoding(pos, index, len(example_embedding[pos])))
    print(pe)

[1.0, 0.0, 1.0, 0.0]
[0.5403023058681398, 0.009999833334166664, 0.999999995, 9.999999999998333e-07]
[-0.4161468365471424, 0.01999866669333308, 0.9999999800000001, 1.9999999999986667e-06]


# Most Commonly Used Function in Positional Encoding

The most commonly used functions in positional encoding are **sine** and **cosine** functions. These functions are used because they provide a continuous and smooth way to encode positional information, allowing the model to generalize well to unseen sequence lengths. Additionally, the periodic nature of sine and cosine functions helps capture relative positions effectively.

The formula for positional encoding is as follows:

For even indices:

$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{\frac{2i}{d}}}\right)$

For odd indices:

$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{\frac{2i}{d}}}\right)$

Where:
- $ pos $ is the position of the token in the sequence.
- $ i $ is the dimension index.
- $ d $ is the embedding dimension.

These functions ensure that the positional encodings are unique for each position and dimension, while also being bounded and differentiable, which is crucial for training neural networks.


In [30]:
import torch

def torch_positional_encoding(vocab_size: int, embedding_dim: int):
    # Create a (vocab_size, 1) position tensor
    position = torch.arange(vocab_size).unsqueeze(1)  # shape: (vocab_size, 1)

    # Create a (1, embedding_dim) dimension tensor
    div_term = torch.exp(torch.arange(0, embedding_dim, 2) * (-math.log(10000.0) / embedding_dim))  # shape: (embedding_dim/2,)

    # Compute the positional encoding matrix
    pe = torch.zeros(vocab_size, embedding_dim)
    pe[:, 0::2] = torch.sin(position * div_term)  # even indices
    pe[:, 1::2] = torch.cos(position * div_term)  # odd indices

    return pe  # shape: (vocab_size, embedding_dim)

# Create an example embedding with 10 dimensions for 5 tokens
# 10 tokens, each token vector is 10-dimensional
VOCAB_SIZE = 5
EMBEDDING_DIM = 4
embedding = torch.nn.Embedding(VOCAB_SIZE, EMBEDDING_DIM)
print(embedding.weight)

# Initialize a tensor with VOCAB_SIZE * EMBEDDING_DIM
pe = torch_positional_encoding(VOCAB_SIZE, EMBEDDING_DIM)
print(pe)

print(embedding.weight + pe)

Parameter containing:
tensor([[ 1.7607, -1.7318, -0.7362, -0.1015],
        [ 0.6675,  1.2420, -0.1391, -0.5978],
        [ 0.3526,  0.2350,  0.6259, -0.1230],
        [ 0.8108,  1.4023,  1.5115, -1.0566],
        [-0.4484,  0.4682, -0.1705,  0.4760]], requires_grad=True)
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998],
        [ 0.1411, -0.9900,  0.0300,  0.9996],
        [-0.7568, -0.6536,  0.0400,  0.9992]])
tensor([[ 1.7607, -0.7318, -0.7362,  0.8985],
        [ 1.5090,  1.7823, -0.1291,  0.4022],
        [ 1.2619, -0.1812,  0.6459,  0.8768],
        [ 0.9519,  0.4123,  1.5415, -0.0571],
        [-1.2052, -0.1854, -0.1305,  1.4752]], grad_fn=<AddBackward0>)
